# MDN Natural Gas — Training Notebook

Mixture Density Network for probabilistic NG return forecasting.

**Pipeline:**
1. Load NG futures OHLCV (HDF5 / Sierra Chart CSV / synthetic stub)
2. Pull EIA weekly underground storage via `NatGasHelper` → compute `ConsensusForecast` surprise → interpolate to daily
3. Merge and build features with `NGFeatureEngine` (deseasonalized returns + Parkinson / GK / YZ vol)
4. Train `MDNNetwork` with `MDNNLLLoss + MDNEntropyRegularizer`
5. Diagnostics: loss curves, component usage, predicted density plots
6. Save weights + scaler for walk-forward notebook

Walk-forward validation is covered in the next notebook.

In [ ]:
import sys
import copy
import json
import warnings
import dataclasses
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')

# --- CTAFlow ---
from CTAFlow.models.deep_learning.mixture import (
    MDNConfig,
    MDNNetwork,
    NGFeatureEngine,
    interpolate_weekly_storage,
)
from CTAFlow.models.deep_learning.training.loss import MDNNLLLoss, MDNEntropyRegularizer

# --- macrOS-Int ---
sys.path.insert(0, r'C:\Users\nicho\PyCharmProjects\macrOS-Int')
from MacrOSINT.data.sources.eia.api_tools import NatGasHelper
from MacrOSINT.models.energy.natgas_storage_forecast import fetch_storage_data, ConsensusForecast

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
config = MDNConfig(
    # Architecture
    n_components=5,
    hidden_dims=[128, 64, 32],
    dropout=0.3,
    use_batch_norm=True,
    activation='silu',
    # Target
    target_horizon=1,           # 1=next-day | 5=weekly | 21=monthly forward return
    deseasonalize_returns=True,
    # Features
    include_fundamentals=True,
    include_cross_asset=False,  # set True when crude/DXY columns are available
    lookback_lags=5,
    vol_windows=[5, 10, 20, 60],
    momentum_windows=[1, 5, 10, 20],
    # Training
    max_epochs=200,
    patience=20,
    learning_rate=1e-3,
    weight_decay=1e-4,
    batch_size=64,
    grad_clip=1.0,
    entropy_weight=0.1,
    lr_scheduler='cosine',
    # Reproducibility
    seed=42,
)

torch.manual_seed(config.seed)
np.random.seed(config.seed)
print(f'Target horizon : {config.target_horizon}d')
print(f'Components     : {config.n_components}')
print(f'Hidden dims    : {config.hidden_dims}')

## 2. Load Price Data

Needs at minimum a `close` column with a DatetimeIndex.  
`open`, `high`, `low` unlock Parkinson / Garman-Klass / Yang-Zhang vol estimators.  
Uncomment the appropriate loader below.

In [ ]:
# --- Option A: HDF5 via CTAFlow DataClient ---
# from CTAFlow.data.data_client import DataClient
# price_df = DataClient().load('NG1')  # daily OHLCV

# --- Option B: Sierra Chart CSV export ---
# from CTAFlow.data.raw_formatting.intraday_manager import read_exported_df
# price_df = read_exported_df(r'path\to\NG1_daily.csv')
# Resample to daily if intraday:
# price_df = price_df.resample('B').agg({'open':'first','high':'max','low':'min','close':'last','volume':'sum'}).dropna()

# --- Option C: Synthetic stub for smoke-testing ---
rng = np.random.RandomState(config.seed)
n = 2500
dates = pd.bdate_range('2015-01-02', periods=n)
prices = 3.0 * np.exp(np.cumsum(rng.normal(0, 0.025, n)))
price_df = pd.DataFrame({
    'open':  prices * (1 + rng.normal(0, 0.003, n)),
    'high':  prices * (1 + np.abs(rng.normal(0, 0.012, n))),
    'low':   prices * (1 - np.abs(rng.normal(0, 0.012, n))),
    'close': prices,
    'volume': rng.lognormal(12, 0.5, n).astype(int),
}, index=dates)

print(f'Price data : {price_df.shape}')
print(f'Date range : {price_df.index[0].date()} → {price_df.index[-1].date()}')
print(f'Columns    : {price_df.columns.tolist()}')
price_df.tail(3)

## 3. EIA Weekly Storage → Consensus Surprise → Daily

`fetch_storage_data()` wraps `NatGasHelper` to return a clean weekly DataFrame with `storage_level` (BCF) and `storage_change` (weekly injection/withdrawal).  

`ConsensusForecast` builds a rolling seasonal blend optimised against the M0-M1 spread reaction (weights ~sea_max: 0.62, sea_mean: 0.28, sea_med: 0.10) and produces a `surprise = actual_change − consensus_est` column — a better proxy than a naive rolling mean.

`interpolate_weekly_storage()` then forward-fills all weekly columns to the daily price calendar.

In [ ]:
START = price_df.index[0].strftime('%Y-%m')
END   = price_df.index[-1].strftime('%Y-%m')

try:
    ng_helper   = NatGasHelper()
    storage_wkly = fetch_storage_data(ng_helper, start=START, end=END)
    print(f'EIA storage loaded : {storage_wkly.shape}')
    print(f'Columns            : {storage_wkly.columns.tolist()}')
    storage_wkly.tail(3)
except Exception as e:
    print(f'EIA fetch failed ({e}) — using synthetic weekly stub')
    weekly_idx = pd.date_range(price_df.index[0], price_df.index[-1], freq='W-FRI')
    level = 2500 + np.cumsum(rng.normal(0, 20, len(weekly_idx)))
    storage_wkly = pd.DataFrame({
        'storage_level':  level,
        'storage_change': np.diff(level, prepend=level[0]),
    }, index=weekly_idx)

In [ ]:
# Fit ConsensusForecast on the full storage_change history
# (uses only data up to each date internally — causal by design)
try:
    cf = ConsensusForecast()
    cf.fit(storage_wkly['storage_change'])
    surprise_df = cf.transform()  # columns: actual, consensus_est, surprise, sea_mean, sea_med, ...
    print(f'ConsensusForecast columns: {surprise_df.columns.tolist()}')

    # Join surprise onto weekly storage
    storage_wkly = storage_wkly.join(
        surprise_df[['consensus_est', 'surprise']],
        how='left'
    )
except Exception as e:
    print(f'ConsensusForecast failed ({e}) — computing rolling 4-week proxy')
    chg = storage_wkly['storage_change']
    storage_wkly['consensus_est'] = chg.rolling(4).mean()
    storage_wkly['surprise']      = chg - storage_wkly['consensus_est']

print(f'Weekly storage columns: {storage_wkly.columns.tolist()}')
storage_wkly.tail(4)

In [ ]:
# Forward-fill all weekly columns to the daily price calendar
storage_daily = interpolate_weekly_storage(
    storage_wkly,
    daily_index=price_df.index,
)
print(f'Storage daily : {storage_daily.shape}')
print(f'Columns       : {storage_daily.columns.tolist()}')
storage_daily.tail(3)

## 4. Merge and Build Features

`NGFeatureEngine` picks up:
- `storage_surprise` → surprise z-score feature (pre-computed above, highest priority)
- `storage_level` → level + 5yr deviation features
- `storage_change` → 4-week rolling mean feature

No back-calculation needed — the columns map directly.

In [ ]:
combined = price_df.join(storage_daily, how='left')
print(f'Combined shape : {combined.shape}')
print(f'Storage NaN%   : {combined[["storage_level","storage_surprise"]].isna().mean().to_dict()}')
combined.tail(3)

In [ ]:
engine   = NGFeatureEngine(config)
features = engine.build(combined)

print(f'Features built : {features.shape}')
print(f'Feature count  : {len(engine.feature_names)}')
print(f'Date range     : {features.index[0].date()} → {features.index[-1].date()}')
print(f'\nTarget stats   : mean={features["target"].mean():.5f}  '
      f'std={features["target"].std():.5f}  '
      f'skew={features["target"].skew():.3f}')
print(f'\nFeature names  :')
for i in range(0, len(engine.feature_names), 6):
    print(' ', engine.feature_names[i:i+6])

## 5. Train / Val Split

75 / 15 / 10 chronological split.  
Test set is held out — do not touch until the walk-forward notebook.

In [ ]:
TRAIN_FRAC = 0.75
VAL_FRAC   = 0.15

n       = len(features)
n_train = int(n * TRAIN_FRAC)
n_val   = int(n * VAL_FRAC)

feat_cols = engine.feature_names
train_df  = features.iloc[:n_train]
val_df    = features.iloc[n_train : n_train + n_val]
test_df   = features.iloc[n_train + n_val :]

X_train = train_df[feat_cols].values.astype(np.float32)
y_train = train_df['target'].values.astype(np.float32)
X_val   = val_df[feat_cols].values.astype(np.float32)
y_val   = val_df['target'].values.astype(np.float32)

# Standardize on train statistics only — no lookahead
mu_X  = X_train.mean(axis=0)
sd_X  = X_train.std(axis=0) + 1e-8
X_train_s = (X_train - mu_X) / sd_X
X_val_s   = (X_val   - mu_X) / sd_X

print(f'Train  : {X_train_s.shape}  {train_df.index[0].date()} → {train_df.index[-1].date()}')
print(f'Val    : {X_val_s.shape}  {val_df.index[0].date()} → {val_df.index[-1].date()}')
print(f'Test   : {len(test_df)} rows (held out)  {test_df.index[0].date()} → {test_df.index[-1].date()}')

In [ ]:
def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    ds = TensorDataset(
        torch.FloatTensor(X).to(DEVICE),
        torch.FloatTensor(y).to(DEVICE),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=False)

train_loader = make_loader(X_train_s, y_train, config.batch_size, shuffle=True)
val_loader   = make_loader(X_val_s,   y_val,   config.batch_size, shuffle=False)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

## 6. Model + Losses

In [ ]:
n_features = len(feat_cols)
model      = MDNNetwork(config, n_features).to(DEVICE)

nll_loss = MDNNLLLoss(reduction='mean')
ent_reg  = MDNEntropyRegularizer(target_entropy_frac=0.5)

optimizer = optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Input dim    : {n_features}')
print(f'Components   : {config.n_components}')
print(f'Trainable    : {total_params:,} params')
print(model)

## 7. Training Loop

In [ ]:
history = {'train_nll': [], 'val_nll': [], 'val_entropy': [], 'val_mean_sigma': []}
best_val_nll = float('inf')
best_state   = None
patience_cnt = 0

for epoch in range(1, config.max_epochs + 1):

    # ---- Train ----
    model.train()
    for X_b, y_b in train_loader:
        optimizer.zero_grad()
        pi, mu, sigma = model(X_b)
        loss = nll_loss(pi, mu, sigma, y_b) + config.entropy_weight * ent_reg(pi)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        optimizer.step()
    scheduler.step()

    # ---- Evaluate ----
    model.eval()
    with torch.no_grad():
        t_nlls, v_nlls, v_ents, v_sigmas = [], [], [], []
        for X_b, y_b in train_loader:
            pi, mu, sigma = model(X_b)
            t_nlls.append(nll_loss(pi, mu, sigma, y_b).item())
        for X_b, y_b in val_loader:
            pi, mu, sigma = model(X_b)
            v_nlls.append(nll_loss(pi, mu, sigma, y_b).item())
            v_ents.append(ent_reg(pi).item())
            v_sigmas.append(sigma.mean().item())

    t_nll = float(np.mean(t_nlls))
    v_nll = float(np.mean(v_nlls))
    history['train_nll'].append(t_nll)
    history['val_nll'].append(v_nll)
    history['val_entropy'].append(float(np.mean(v_ents)))
    history['val_mean_sigma'].append(float(np.mean(v_sigmas)))

    # ---- Early stopping ----
    if v_nll < best_val_nll:
        best_val_nll = v_nll
        best_state   = copy.deepcopy(model.state_dict())
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 10 == 0:
        lr = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d} | '
              f'Train NLL: {t_nll:.4f} | '
              f'Val NLL: {v_nll:.4f} | '
              f'σ̄: {history["val_mean_sigma"][-1]:.4f} | '
              f'LR: {lr:.2e} | '
              f'Patience: {patience_cnt}/{config.patience}')

    if patience_cnt >= config.patience:
        print(f'Early stopping at epoch {epoch}')
        break

model.load_state_dict(best_state)
print(f'\nBest val NLL : {best_val_nll:.4f}')

## 8. Training Diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, len(history['train_nll']) + 1)

axes[0].plot(ep, history['train_nll'], label='Train')
axes[0].plot(ep, history['val_nll'],   label='Val')
axes[0].set_title('NLL Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(ep, history['val_entropy'], color='darkorange')
axes[1].set_title('Entropy Reg (val)'); axes[1].set_xlabel('Epoch')
axes[1].axhline(0, color='k', lw=0.8, linestyle='--')

axes[2].plot(ep, history['val_mean_sigma'], color='green')
axes[2].set_title('Mean σ̄ (val)'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

## 9. Component Usage — Validation Set

In [ ]:
model.eval()
all_pi, all_mu, all_sigma = [], [], []

with torch.no_grad():
    for X_b, _ in val_loader:
        pi, mu, sigma = model(X_b)
        all_pi.append(pi.cpu().numpy())
        all_mu.append(mu.cpu().numpy())
        all_sigma.append(sigma.cpu().numpy())

pi_val    = np.concatenate(all_pi)
mu_val    = np.concatenate(all_mu)
sigma_val = np.concatenate(all_sigma)

avg_pi    = pi_val.mean(axis=0)
avg_mu    = mu_val.mean(axis=0)
avg_sigma = sigma_val.mean(axis=0)

print(f'{"k":>3}  {"π̄":>8}  {"μ̄":>10}  {"σ̄":>10}')
print('-' * 38)
for k in range(config.n_components):
    flag = '  ← underutilized' if avg_pi[k] < 0.05 else ''
    print(f'{k:>3}  {avg_pi[k]:>8.3f}  {avg_mu[k]:>+10.5f}  {avg_sigma[k]:>10.5f}{flag}')

dead = (avg_pi < 0.05).sum()
if dead:
    print(f'\n⚠  {dead} component(s) underutilized — consider n_components={config.n_components - dead}')

# Bar chart
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(config.n_components), avg_pi, color='steelblue', edgecolor='black')
ax.axhline(1 / config.n_components, color='r', linestyle='--', label='uniform')
ax.set_xlabel('Component'); ax.set_ylabel('Average π'); ax.set_title('Component Utilization')
ax.legend(); plt.tight_layout(); plt.show()

## 10. Predicted Density — High-Uncertainty Days

In [ ]:
X_val_t = torch.FloatTensor(X_val_s).to(DEVICE)

model.eval()
with torch.no_grad():
    pi_t, mu_t, sigma_t = model(X_val_t)

pi_np    = pi_t.cpu().numpy()
mu_np    = mu_t.cpu().numpy()
sigma_np = sigma_t.cpu().numpy()

# Mixture mean and std
mix_mean = (pi_np * mu_np).sum(axis=1)
mix_var  = (pi_np * (sigma_np**2 + mu_np**2)).sum(axis=1) - mix_mean**2
mix_std  = np.sqrt(np.clip(mix_var, 0, None))

# 4 highest-uncertainty days
top_idx  = np.argsort(mix_std)[-4:]
y_lo     = min(y_val.min(), mix_mean.min()) - 3 * mix_std.max()
y_hi     = max(y_val.max(), mix_mean.max()) + 3 * mix_std.max()
y_grid   = np.linspace(y_lo, y_hi, 400)
y_grid_t = torch.FloatTensor(y_grid).to(DEVICE)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, idx in zip(axes, top_idx):
    with torch.no_grad():
        density = model.predict_density(X_val_t[[idx]], y_grid_t).cpu().numpy()[0]
    ax.plot(y_grid, density, lw=1.5)
    ax.fill_between(y_grid, density, alpha=0.15)
    ax.axvline(y_val[idx],   color='red',   lw=1.2, linestyle='--', label='actual')
    ax.axvline(mix_mean[idx], color='green', lw=1.2, linestyle=':',  label='E[y]')
    ax.set_title(f'{val_df.index[idx].date()}\nσ̄={mix_std[idx]:.4f}')
    ax.legend(fontsize=7); ax.set_xlabel('Return')

plt.suptitle('Predicted Mixture Density — Highest Uncertainty Days', y=1.02)
plt.tight_layout()
plt.show()

## 11. Prediction Summary on Validation Set

In [ ]:
# Monte Carlo quantiles from the mixture
with torch.no_grad():
    samples = model.sample(X_val_t, n_samples=2000).cpu().numpy()  # (N_val, 2000)

pred_df = pd.DataFrame(index=val_df.index)
pred_df['actual']    = y_val
pred_df['pred_mean'] = mix_mean
pred_df['pred_std']  = mix_std
for q in [5, 25, 50, 75, 95]:
    pred_df[f'q{q:02d}'] = np.percentile(samples, q, axis=1)

# Directional accuracy
mask     = np.abs(y_val) > 1e-6
hit_rate = (np.sign(mix_mean[mask]) == np.sign(y_val[mask])).mean()

# Interval coverage
coverage_90 = ((y_val >= pred_df['q05'].values) & (y_val <= pred_df['q95'].values)).mean()

print(f'Directional accuracy : {hit_rate:.1%}')
print(f'90% interval coverage: {coverage_90:.1%}  (target ~90%)')

# Quick time-series plot
fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(pred_df.index, pred_df['q05'], pred_df['q95'], alpha=0.2, label='90% CI')
ax.fill_between(pred_df.index, pred_df['q25'], pred_df['q75'], alpha=0.3, label='50% CI')
ax.plot(pred_df.index, pred_df['actual'],    color='black', lw=0.7, alpha=0.8, label='actual')
ax.plot(pred_df.index, pred_df['pred_mean'], color='blue',  lw=0.8, alpha=0.7, label='E[y]')
ax.set_title('Validation: Actual vs Predicted Quantiles')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()
pred_df.describe()

## 12. Save Model, Scaler, and Features

In [ ]:
SAVE_DIR = Path(r'C:\Users\nicho\PycharmProjects\CTAFlow\outputs\mdn_natgas')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Model weights
torch.save(model.state_dict(), SAVE_DIR / 'mdn_natgas.pth')

# Train-set scaler (mu, sd, feature names) — used per-fold in walk-forward
scaler = {
    'mu_X':          mu_X.tolist(),
    'sd_X':          sd_X.tolist(),
    'feature_names': feat_cols,
}
with open(SAVE_DIR / 'scaler.json', 'w') as f:
    json.dump(scaler, f)

# Config
with open(SAVE_DIR / 'config.json', 'w') as f:
    json.dump(dataclasses.asdict(config), f, indent=2)

# Full feature DataFrame (pickled for walk-forward notebook)
features.to_pickle(SAVE_DIR / 'features.pkl')

print(f'Saved to {SAVE_DIR}')
for p in sorted(SAVE_DIR.iterdir()):
    print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

## Next: Walk-Forward Validation

Load `features.pkl` and `config.json` from the output directory, then run expanding-window folds:

| Window | Size |
|---|---|
| Train | grows from `config.train_window` (504d) |
| Val   | fixed `config.val_window` (63d) |
| Test  | `config.test_window` step (21d) |

Each fold: fit train-only scaler → retrain `MDNNetwork` from scratch → collect OOS predictions.  
Aggregate metrics: NLL, directional accuracy, signal Sharpe, PIT calibration, tail breach rates.